5.2 Installation


In [1]:
pip install pandas numpy matplotlib seaborn scikit-learn openpyxl


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Load public dataset
df = pd.read_excel("Online Retail.xlsx")

# Validate required columns
required = ["InvoiceNo", "InvoiceDate", "CustomerID",
            "Quantity", "UnitPrice"]
missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Clean and validate
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df = df.dropna(subset=["CustomerID", "InvoiceDate"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df = df[~df["InvoiceNo"].str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df["LineValue"] = df["Quantity"] * df["UnitPrice"]

# Build RFM table
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date-x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("LineValue", "sum")
).reset_index()

# Transform and scale
features = rfm[["Recency", "Frequency", "Monetary"]].copy()
features["Frequency"] = np.log1p(features["Frequency"])
features["Monetary"] = np.log1p(features["Monetary"])
scaler = StandardScaler()
X = scaler.fit_transform(features)

# Compare K values
results = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    results.append((k, km.inertia_, silhouette_score(X, labels)))

scores = pd.DataFrame(results, columns=["k", "inertia", "silhouette"])
print(scores)

# Fit a candidate final model; inspect profiles before approval
chosen_k = int(scores.loc[scores["silhouette"].idxmax(), "k"])
model = KMeans(n_clusters=chosen_k, random_state=42, n_init=10)
rfm["Cluster"] = model.fit_predict(X)

profile = rfm.groupby("Cluster").agg(
    Customers=("CustomerID", "count"),
    Median_Recency=("Recency", "median"),
    Median_Frequency=("Frequency", "median"),
    Median_Monetary=("Monetary", "median")
).round(2)

rfm.to_csv("customer_segments.csv", index=False)
profile.to_csv("segment_profiles.csv")
print(profile)


   k      inertia  silhouette
0  2  6872.757289    0.406390
1  3  4298.644789    0.415646
2  4  3242.054949    0.380497
3  5  2760.927874    0.343331
4  6  2397.223499    0.330665
5  7  2166.671613    0.299834
6  8  1960.504884    0.300510
         Customers  Median_Recency  Median_Frequency  Median_Monetary
Cluster                                                              
0             2036            46.0               2.0           509.29
1             1323            17.0               7.0          2575.38
2              979           253.0               1.0           291.05
